In [1]:
!pip install openai


[notice] A new release of pip is available: 24.1.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
import pandas as pd
import re
import json
import asyncio

env_path = '.env'
load_dotenv(dotenv_path=env_path)

api_key = os.environ.get("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OpenRouter API key not found. Please set OPENROUTER_API_KEY in your .env file.")

In [3]:
df = pd.read_csv("../scraper/final_dataset.csv")

df

,author,body_text,category,channel,error,link,published_date,scraped_at,title
0,Mikhael Gewati,KOMPAS.com- Dari hamparan pesisir timur hingga...,finance,ekbis,NaN,https://money.kompas.com/read/2025/11/22/09091...,"Kompas.com, 22 November 2025, 09:09 WIB",2025-11-22T16:54:39.469010,"Astra Bergerak Bersama Anak Bangsa, Membangun ..."
1,NaN,BANKIndonesia telah menurunkan suku bunga acua...,finance,ekbis,NaN,https://money.kompas.com/read/2025/11/22/08434...,"Kompas.com, 22 November 2025, 08:43 WIB",2025-11-22T16:54:37.215084,"Jeratan ""Special Rate"": Pasar Ganda Perbankan"
2,Nur Jamal Shaid,"JAKARTA, KOMPAS.com- Harga emas batangan yang ...",finance,ekbis,NaN,https://money.kompas.com/read/2025/11/22/07543...,"Kompas.com, Diperbarui 22/11/2025, 07:55 WIB",2025-11-22T16:54:36.424523,Harga Emas di Pegadaian 22 November 2025: Gale...
3,"Reni Susanti,","BANDUNG, KOMPAS.com– Vaksin HPV NusaGard, vaks...",finance,ekbis,NaN,https://money.kompas.com/read/2025/11/21/21373...,"Kompas.com, 21 November 2025, 21:37 WIB",2025-11-22T16:54:38.680471,Vaksin HPV Produksi Bio Farma Resmi Kantongi S...
4,Sakina Rakhma Diah Setiawan,"JAKARTA, KOMPAS.com -Iklim usaha yang sehat ti...",finance,ekbis,NaN,https://money.kompas.com/read/2025/11/21/21330...,"Kompas.com, 21 November 2025, 21:33 WIB",2025-11-22T16:54:42.749883,KPPU Soroti Tiga Area Risiko Pelanggaran dalam...
...,...,...,...,...,...,...,...,...,...
4013,Nafilah Sri Sagita K -detikHealth,Viral curhat wanita di TikTok sebagai pasangan...,berita-detikhealth,health,NaN,https://health.detik.com/berita-detikhealth/d-...,"Kamis, 20 Nov 2025 16:00 WIB",2025-11-22T13:08:38.505461,Viral Day 3 Nikah Berujung Masuk RS usai Honey...
4014,Sarah Oktaviani Alam -detikHealth,"Seorang YouTuber di Amerika Serikat, Andrew Fe...",berita-detikhealth,health,NaN,https://health.detik.com/berita-detikhealth/d-...,"Kamis, 20 Nov 2025 15:01 WIB",2025-11-22T13:08:39.834112,Youtuber Ikut Eksperimen 'Puasa' Sosmed 30 Har...
4015,"Dinda Islami, detikTV -detikHealth",Menko Pangan Zulhas ingin adanya penanda label...,detiktv,health,NaN,https://health.detik.com/detiktv/d-8220147/vid...,"Kamis, 20 Nov 2025 15:00 WIB",2025-11-22T13:08:41.200387,Video: Menko Pangan Mau Ada Label 'Tinggi Gula...
4016,Nafilah Sri Sagita K -detikHealth,"Nama Yasika Aulia Ramadhani, perempuan berusia...",berita-detikhealth,health,NaN,https://health.detik.com/berita-detikhealth/d-...,"Kamis, 20 Nov 2025 11:43 WIB",2025-11-22T13:08:42.562160,Kepala BGN soal Anak Waka DPRD Sulsel Punya 41...


In [17]:
df["body_text"] = df["body_text"].apply(
    lambda x: re.sub(r"^[^—–-]*[—–-]\s*", "", x)
)

# Jangan run code ini sebelum csv dibaca ulang

In [4]:
df

,author,body_text,category,channel,error,link,published_date,scraped_at,title
0,Mikhael Gewati,KOMPAS.com- Dari hamparan pesisir timur hingga...,finance,ekbis,NaN,https://money.kompas.com/read/2025/11/22/09091...,"Kompas.com, 22 November 2025, 09:09 WIB",2025-11-22T16:54:39.469010,"Astra Bergerak Bersama Anak Bangsa, Membangun ..."
1,NaN,BANKIndonesia telah menurunkan suku bunga acua...,finance,ekbis,NaN,https://money.kompas.com/read/2025/11/22/08434...,"Kompas.com, 22 November 2025, 08:43 WIB",2025-11-22T16:54:37.215084,"Jeratan ""Special Rate"": Pasar Ganda Perbankan"
2,Nur Jamal Shaid,"JAKARTA, KOMPAS.com- Harga emas batangan yang ...",finance,ekbis,NaN,https://money.kompas.com/read/2025/11/22/07543...,"Kompas.com, Diperbarui 22/11/2025, 07:55 WIB",2025-11-22T16:54:36.424523,Harga Emas di Pegadaian 22 November 2025: Gale...
3,"Reni Susanti,","BANDUNG, KOMPAS.com– Vaksin HPV NusaGard, vaks...",finance,ekbis,NaN,https://money.kompas.com/read/2025/11/21/21373...,"Kompas.com, 21 November 2025, 21:37 WIB",2025-11-22T16:54:38.680471,Vaksin HPV Produksi Bio Farma Resmi Kantongi S...
4,Sakina Rakhma Diah Setiawan,"JAKARTA, KOMPAS.com -Iklim usaha yang sehat ti...",finance,ekbis,NaN,https://money.kompas.com/read/2025/11/21/21330...,"Kompas.com, 21 November 2025, 21:33 WIB",2025-11-22T16:54:42.749883,KPPU Soroti Tiga Area Risiko Pelanggaran dalam...
...,...,...,...,...,...,...,...,...,...
4013,Nafilah Sri Sagita K -detikHealth,Viral curhat wanita di TikTok sebagai pasangan...,berita-detikhealth,health,NaN,https://health.detik.com/berita-detikhealth/d-...,"Kamis, 20 Nov 2025 16:00 WIB",2025-11-22T13:08:38.505461,Viral Day 3 Nikah Berujung Masuk RS usai Honey...
4014,Sarah Oktaviani Alam -detikHealth,"Seorang YouTuber di Amerika Serikat, Andrew Fe...",berita-detikhealth,health,NaN,https://health.detik.com/berita-detikhealth/d-...,"Kamis, 20 Nov 2025 15:01 WIB",2025-11-22T13:08:39.834112,Youtuber Ikut Eksperimen 'Puasa' Sosmed 30 Har...
4015,"Dinda Islami, detikTV -detikHealth",Menko Pangan Zulhas ingin adanya penanda label...,detiktv,health,NaN,https://health.detik.com/detiktv/d-8220147/vid...,"Kamis, 20 Nov 2025 15:00 WIB",2025-11-22T13:08:41.200387,Video: Menko Pangan Mau Ada Label 'Tinggi Gula...
4016,Nafilah Sri Sagita K -detikHealth,"Nama Yasika Aulia Ramadhani, perempuan berusia...",berita-detikhealth,health,NaN,https://health.detik.com/berita-detikhealth/d-...,"Kamis, 20 Nov 2025 11:43 WIB",2025-11-22T13:08:42.562160,Kepala BGN soal Anak Waka DPRD Sulsel Punya 41...


In [5]:
client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    default_headers={"HTTP-Referer": "http://localhost:5000"}
)

In [6]:
ner_prompts = '''Here is a paragraph. Please extract all the named entities in the paragraph with the following format: [{{"entity": Entity Name 1, "type": Entity Type 1}}, {{"entity": Entity Name 2, "type": Entity Type 2}}, ...].

Only extract entities that match these types:
- CRD: Cardinal (numbers that don't fall into other categories)
- DAT: Date (specific dates, days, months, years)
- EVT: Event (named events like conferences, wars, sports events)
- FAC: Facility (buildings, airports, highways, bridges)
- GPE: Geopolitical Entity (countries, cities, states)
- LAW: Law Entity (laws, legal documents, like Undang-Undang)
- LOC: Location (non-GPE locations, mountain ranges, bodies of water)
- MON: Money (monetary values)
- NOR: Political Organization (political parties, movements)
- ORD: Ordinal (first, second, third, etc.)
- ORG: Organization (companies, agencies, institutions)
- PER: Person (people, including fictional)
- PRC: Percent (percentage values)
- PRD: Product (objects, vehicles, foods, etc., not services)
- QTY: Quantity (measurements, weights, distances)
- REG: Religion (religions, religious groups)
- TIM: Time (times smaller than a day)
- WOA: Work of Art (titles of books, songs, paintings)
- LAN: Language (named languages)

Paragraph: {para}

Named Entities (JSON format only):''' 
summary_prompts = "Here is a paragraph. Please give a summary of the paragraph. \n\nParagraph: {para}\n\n Summary: "

In [7]:
user_prompts = df['body_text']
system_prompt = "You are the smartest AI assistant"

In [8]:
processed_ner_prompts = []
processed_summary_prompts = []

for i in user_prompts:
    processed_ner_prompts.append(ner_prompts.format(para=i))
    processed_summary_prompts.append(summary_prompts.format(para=i))

In [9]:
ner_res = []
summary_res = []

In [10]:
sema = asyncio.Semaphore(10)

In [11]:
async def run_prompt(processed_prompt, idx, status):
    async with sema:
        print(idx, status)
        
        final_prompt = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": processed_prompt}
        ]

        response = await client.chat.completions.create(
            model="qwen/qwen3-30b-a3b-instruct-2507",
            messages=final_prompt
        )

        return {
            "prompt": processed_prompt,
            "answer": response.choices[0].message.content
        }

In [12]:
async def run_ner_prompts():
    tasks = [run_prompt(p, i, 'ner') for i, p in enumerate(processed_ner_prompts)]
    results = await asyncio.gather(*tasks)
    return results

async def run_summary_prompts():
    tasks = [run_prompt(p, i, 'summary') for i, p in enumerate(processed_summary_prompts)]
    results = await asyncio.gather(*tasks)
    return results

async def main():
    results = await asyncio.gather(
        run_ner_prompts(),
        # run_summary_prompts()
    )
    return results

In [13]:
res = await main()

0 ner
1 ner
2 ner
3 ner
4 ner
5 ner
6 ner
7 ner
8 ner
9 ner
10 ner
11 ner
12 ner
13 ner
14 ner
15 ner
16 ner
17 ner
18 ner
19 ner
20 ner
21 ner
22 ner
23 ner
24 ner
25 ner
26 ner
27 ner
28 ner
29 ner
30 ner
31 ner
32 ner
33 ner
34 ner
35 ner
36 ner
37 ner
38 ner
39 ner
40 ner
41 ner
42 ner
43 ner
44 ner
45 ner
46 ner
47 ner
48 ner
49 ner
50 ner
51 ner
52 ner
53 ner
54 ner
55 ner
56 ner
57 ner
58 ner
59 ner
60 ner
61 ner
62 ner
63 ner
64 ner
65 ner
66 ner
67 ner
68 ner
69 ner
70 ner
71 ner
72 ner
73 ner
74 ner
75 ner
76 ner
77 ner
78 ner
79 ner
80 ner
81 ner
82 ner
83 ner
84 ner
85 ner
86 ner
87 ner
88 ner
89 ner
90 ner
91 ner
92 ner
93 ner
94 ner
95 ner
96 ner
97 ner
98 ner
99 ner
100 ner
101 ner
102 ner
103 ner
104 ner
105 ner
106 ner
107 ner
108 ner
109 ner
110 ner
111 ner
112 ner
113 ner
114 ner
115 ner
116 ner
117 ner
118 ner
119 ner
120 ner
121 ner
122 ner
123 ner
124 ner
125 ner
126 ner
127 ner
128 ner
129 ner
130 ner
131 ner
132 ner
133 ner
134 ner
135 ner
136 ner
137 ner
138 ne

In [14]:
files_length = len(os.listdir())

with open(f"{files_length}.json", "w") as f:
    json.dump(res, f, indent=2)